# Train Isolation Forest — Phát hiện bất thường hóa đơn thuốc

**Pipeline:**
1. Upload `training_data_v2.csv`
2. Feature engineering từ dữ liệu thô
3. Train Isolation Forest chỉ trên normal rows
4. Đánh giá kết quả
5. Lưu `isolation_forest.pkl` + `scaler.pkl` → tải về dùng trong FastAPI

> **Lưu ý:** Model train trên normal rows, KHÔNG dùng `severity`/`flags` vì đây là output của rule engine — không có khi deploy thật.

## Bước 1 — Cài thư viện & upload data

In [ ]:
# Cài thư viện cần thiết
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
from google.colab import files

print('Upload file training_data_v2.csv')
uploaded = files.upload()
# Sau khi upload xong, file sẽ nằm ở /content/training_data_v2.csv

## Bước 2 — Load data & kiểm tra

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('training_data_v2.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print('=== Phân bố nhãn ===')
vc = df['is_anomaly'].value_counts()
print(f'  Normal  (0): {vc.get(0, 0):>6} rows ({vc.get(0,0)/len(df)*100:.1f}%)')
print(f'  Anomaly (1): {vc.get(1, 0):>6} rows ({vc.get(1,0)/len(df)*100:.1f}%)')
print()
print('=== Null counts ===')
print(df[['dangBaoChe','donViTinh','price_ref','ngayKK_KKL_dt','loaiGia']].isnull().sum())
print()
df.head(3)

## Bước 3 — Feature Engineering

In [ ]:
# ── Chuẩn hóa text trước khi classify ──
df['dangBaoChe_norm'] = df['dangBaoChe'].fillna('').str.lower().str.strip()
df['donViTinh_norm']  = df['donViTinh'].fillna('').str.lower().str.strip()

# ── Phân loại dạng bào chế ──
# is_injection: ưu tiên dangBaoChe, dùng donViTinh làm fallback
INJ_DBC = 'tiêm|truyền|bột pha tiêm|nhũ tương tiêm|dung dịch tiêm|bột đông khô'
INJ_DVT = ['ống', 'lọ', 'ong', 'lo']
TAB_DBC = 'viên|nén|nang|vỉ'
SYR_DBC = 'siro|hỗn dịch|cốm pha|bột pha hỗn dịch'

df['is_injection'] = (
    df['dangBaoChe_norm'].str.contains(INJ_DBC, case=False) |
    df['donViTinh_norm'].isin(INJ_DVT)
).astype(int)

df['is_tablet'] = df['dangBaoChe_norm'].str.contains(TAB_DBC, case=False).astype(int)
df['is_syrup']  = df['dangBaoChe_norm'].str.contains(SYR_DBC, case=False).astype(int)

# ── Loại giá (nhập khẩu → kiểm soát chặt hơn) ──
df['is_import'] = df['loaiGia'].fillna('').str.contains('nhập khẩu', case=False).astype(int)

# ── Price features ──
df['price_ratio'] = (df['donGia'] / df['price_ref'].replace(0, np.nan)).fillna(1.0)
# Clip outlier cực đoan (> 10x là data noise)
df['price_ratio'] = df['price_ratio'].clip(0, 10)

# ── Log transform (giá và tiền phân phối lệch rất mạnh) ──
df['log_donGia']   = np.log1p(df['donGia'])
df['log_thanh']    = np.log1p(df['thanhTien'])
df['log_soLuong']  = np.log1p(df['soLuong'])

# ── Metadata quality ──
df['date_missing'] = df['ngayKK_KKL_dt'].isna().astype(int)

# ── Feature list cuối ──
# LƯU Ý: KHÔNG dùng severity, flags, reason_code
# Vì đây là output của rule engine — khi deploy thật hóa đơn mới sẽ không có
FEATURES = [
    'price_ratio',    # giá so với tham chiếu (quan trọng nhất)
    'log_donGia',     # đơn giá (log scale)
    'log_thanh',      # thành tiền (log scale)
    'log_soLuong',    # số lượng (log scale)
    'soLuong',        # số lượng gốc (bắt high_qty rule)
    'key_quality',    # chất lượng match với drug master (2 hoặc 3)
    'is_injection',   # dạng tiêm/truyền → ngưỡng kiểm soát chặt
    'is_tablet',      # dạng viên
    'is_syrup',       # dạng siro
    'is_import',      # thuốc nhập khẩu
    'date_missing',   # thiếu ngày kê khai (corr=-0.11 với anomaly)
]

print('Features được dùng:')
for f in FEATURES:
    print(f'  {f}')

print()
print('Kiểm tra null sau feature engineering:')
print(df[FEATURES].isnull().sum())
print()
print('Stats tóm tắt:')
df[FEATURES].describe().round(3)

## Bước 4 — Chuẩn bị train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[FEATURES].fillna(0).values
y = df['is_anomaly'].values

# stratify để giữ tỉ lệ anomaly/normal trong cả train lẫn test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale: quan trọng cho Isolation Forest
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit trên train
X_test_s  = scaler.transform(X_test)        # chỉ transform trên test

# Tách normal rows để train IF đúng cách
X_train_normal = X_train_s[y_train == 0]
X_train_anomaly = X_train_s[y_train == 1]

print('=== Phân bố sau split ===')
print(f'Train total : {len(X_train):>6}')
print(f'  Normal    : {len(X_train_normal):>6}  ({len(X_train_normal)/len(X_train)*100:.1f}%)')
print(f'  Anomaly   : {len(X_train_anomaly):>6}  ({len(X_train_anomaly)/len(X_train)*100:.1f}%)')
print(f'Test total  : {len(X_test):>6}')
print(f'  Normal    : {(y_test==0).sum():>6}')
print(f'  Anomaly   : {(y_test==1).sum():>6}')
print()
print('IF sẽ fit trên', len(X_train_normal), 'normal rows')

## Bước 5 — Train Isolation Forest

**Tại sao fit chỉ trên normal rows?**

Isolation Forest học "hình dạng" của dữ liệu bình thường. Khi gặp điểm mới nằm ngoài vùng đó → flagged anomaly. Nếu feed cả anomaly vào, model học luôn pattern của anomaly → kém phân biệt hơn.

In [ ]:
from sklearn.ensemble import IsolationForest
import time

# contamination = tỉ lệ ước tính anomaly trong production
# Dùng 0.5 vì dataset train thiên về anomaly nhiều
# Trong production thật tỉ lệ thấp hơn, nhưng 0.5 cho F1-anomaly tốt nhất
CONTAMINATION = 0.5

print(f'Training Isolation Forest...')
print(f'  n_estimators = 400')
print(f'  contamination = {CONTAMINATION}')
print(f'  fit trên {len(X_train_normal)} normal rows')
print()

t0 = time.time()
model = IsolationForest(
    n_estimators=400,
    contamination=CONTAMINATION,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_normal)
print(f'Done! ({time.time()-t0:.1f}s)')

## Bước 6 — Đánh giá kết quả

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Predict: -1 = anomaly, 1 = normal
raw_preds  = model.predict(X_test_s)
preds      = np.where(raw_preds == -1, 1, 0)   # chuyển sang 0/1
anom_score = -model.decision_function(X_test_s) # score cao = bất thường

print('=' * 55)
print('CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, preds, target_names=['Normal', 'Anomaly']))

auc = roc_auc_score(y_test, anom_score)
print(f'ROC-AUC Score: {auc:.4f}')
print()

cm = confusion_matrix(y_test, preds)
print('Confusion Matrix:')
print(f'  True Normal  (TN): {cm[0,0]}  — normal đúng')
print(f'  False Anomaly(FP): {cm[0,1]}  — normal bị báo nhầm là anomaly')
print(f'  Missed Anomaly(FN): {cm[1,0]}  — anomaly bị bỏ sót ← quan trọng')
print(f'  True Anomaly (TP): {cm[1,1]}  — anomaly bắt đúng')

# Vẽ confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Normal', 'Pred: Anomaly'],
            yticklabels=['Actual: Normal', 'Actual: Anomaly'])
axes[0].set_title('Confusion Matrix')

# Distribution of anomaly scores
axes[1].hist(anom_score[y_test==0], bins=50, alpha=0.6, label='Normal', color='steelblue')
axes[1].hist(anom_score[y_test==1], bins=50, alpha=0.6, label='Anomaly', color='tomato')
axes[1].set_xlabel('Anomaly Score (cao = bất thường hơn)')
axes[1].set_ylabel('Count')
axes[1].set_title('Phân bố Anomaly Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Đã lưu evaluation_plots.png')

In [ ]:
# Phân tích theo severity để biết model bắt được loại nào tốt nhất
df_test = df.iloc[X_test.shape[0] * 0:].copy()  # placeholder

# Rebuild test dataframe với predictions
_, df_te_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y
)
df_te = df.iloc[df_te_idx].copy()
df_te['pred_anomaly'] = preds
df_te['anomaly_score'] = anom_score

print('=== Detection rate theo severity ===')
for sev in sorted(df_te['severity'].unique()):
    sub = df_te[df_te['severity'] == sev]
    if len(sub) == 0:
        continue
    caught = sub['pred_anomaly'].sum()
    label = {0: 'Normal', 1: 'Cảnh báo nhẹ', 2: 'Cần kiểm tra', 3: 'Nghiêm trọng'}.get(sev, str(sev))
    print(f'  Severity {sev} ({label:20}): {caught:>4}/{len(sub):>4}  = {caught/len(sub)*100:>5.1f}%')

print()
print('=== Top 10 dòng anomaly score cao nhất ===')
top10 = df_te.nlargest(10, 'anomaly_score')[[
    'tenThuoc', 'soLuong', 'donGia', 'price_ref', 'anomaly_score', 'severity', 'is_anomaly'
]]
pd.set_option('display.max_colwidth', 30)
print(top10.to_string(index=False))

## Bước 7 — Lưu model

In [ ]:
import pickle
import os

os.makedirs('saved_model', exist_ok=True)

# Lưu model
with open('saved_model/isolation_forest.pkl', 'wb') as f:
    pickle.dump(model, f)

# Lưu scaler (BẮT BUỘC dùng cùng scaler khi predict)
with open('saved_model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Lưu feature list (để predict đúng thứ tự)
with open('saved_model/features.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

# Lưu metadata
import json
metadata = {
    'model_type': 'IsolationForest',
    'n_estimators': 400,
    'contamination': CONTAMINATION,
    'trained_on': 'normal_rows_only',
    'n_train_normal': int(len(X_train_normal)),
    'features': FEATURES,
    'f1_anomaly_test': float(f1_score(y_test, preds, pos_label=1)),
    'f1_normal_test':  float(f1_score(y_test, preds, pos_label=0)),
    'roc_auc':         float(auc),
}
with open('saved_model/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('=== Files đã lưu ===')
for fname in os.listdir('saved_model'):
    size = os.path.getsize(f'saved_model/{fname}')
    print(f'  saved_model/{fname}  ({size/1024:.1f} KB)')

print()
print('Metadata:')
print(json.dumps(metadata, indent=2, ensure_ascii=False))

In [ ]:
# Nén và tải về máy
import shutil
from google.colab import files

shutil.make_archive('pharma_model', 'zip', 'saved_model')
print('Tải về pharma_model.zip...')
files.download('pharma_model.zip')
# Giải nén ra → copy isolation_forest.pkl + scaler.pkl + features.pkl
# vào folder model/ trong project FastAPI

## Bước 8 — Kiểm tra predict 1 dòng mẫu (smoke test)

In [ ]:
# Hàm predict dùng lại y chang trong FastAPI pipeline
def predict_item(item: dict, model, scaler, features: list) -> dict:
    """
    item: dict với các key từ hóa đơn sau OCR + drug_matcher
    Trả về: {'is_anomaly': bool, 'anomaly_score': float, 'label': str}
    """
    don_gia   = float(item.get('donGia', 0) or 0)
    price_ref = float(item.get('price_ref', 0) or 0)
    thanh     = float(item.get('thanhTien', 0) or 0)
    so_luong  = float(item.get('soLuong', 0) or 0)
    dbc       = str(item.get('dangBaoChe', '')).lower()
    dvt       = str(item.get('donViTinh', '')).lower()
    loai_gia  = str(item.get('loaiGia', '')).lower()
    ngay_kk   = item.get('ngayKK_KKL_dt', None)
    key_q     = int(item.get('key_quality', 3) or 3)

    INJ_DBC = 'tiêm|truyền|bột pha tiêm|nhũ tương tiêm|dung dịch tiêm'
    INJ_DVT = ['ống', 'lọ', 'ong', 'lo']
    TAB_DBC = 'viên|nén|nang|vỉ'
    SYR_DBC = 'siro|hỗn dịch|cốm pha'

    import re
    row = {
        'price_ratio':  (don_gia / price_ref) if price_ref > 0 else 1.0,
        'log_donGia':   np.log1p(don_gia),
        'log_thanh':    np.log1p(thanh),
        'log_soLuong':  np.log1p(so_luong),
        'soLuong':      so_luong,
        'key_quality':  key_q,
        'is_injection': int(bool(re.search(INJ_DBC, dbc)) or dvt in INJ_DVT),
        'is_tablet':    int(bool(re.search(TAB_DBC, dbc))),
        'is_syrup':     int(bool(re.search(SYR_DBC, dbc))),
        'is_import':    int('nhập khẩu' in loai_gia),
        'date_missing': int(ngay_kk is None or str(ngay_kk) == 'nan'),
    }

    X = np.array([[row[f] for f in features]])
    X = np.clip(X, None, None)  # safety
    X_s = scaler.transform(X)

    pred  = model.predict(X_s)[0]          # -1 = anomaly
    score = float(-model.decision_function(X_s)[0])  # cao = bất thường hơn
    # Normalize về 0-1
    score_norm = min(1.0, max(0.0, (score + 0.5)))

    return {
        'is_anomaly_ml': bool(pred == -1),
        'anomaly_score': round(score_norm, 3),
        'label': 'ANOMALY' if pred == -1 else 'NORMAL'
    }


# ── TEST với hóa đơn HD-000012 từ ảnh thật ──
print('=== Smoke test: HD-000012 ===')
test_items = [
    {'tenThuoc': 'Scandonest 3% Plain', 'soLuong': 12, 'donViTinh': 'ống',
     'dangBaoChe': 'Dung dịch tiêm', 'donGia': 17100, 'thanhTien': 205200,
     'price_ref': 17100, 'loaiGia': 'KK trong nước', 'key_quality': 3},

    {'tenThuoc': 'Spirastad 1.5 M.I.U', 'soLuong': 8, 'donViTinh': 'Viên',
     'dangBaoChe': 'Viên nén', 'donGia': 4300, 'thanhTien': 34400,
     'price_ref': 4300, 'loaiGia': 'KK trong nước', 'key_quality': 3},

    {'tenThuoc': 'Ofloxacin', 'soLuong': 16, 'donViTinh': 'Lọ',
     'dangBaoChe': 'Dung dịch tiêm truyền', 'donGia': 95000, 'thanhTien': 1520000,
     'price_ref': 60000, 'loaiGia': 'KK nhập khẩu', 'key_quality': 3},

    {'tenThuoc': 'Glucose 10%', 'soLuong': 7, 'donViTinh': 'Chai',
     'dangBaoChe': 'Dung dịch truyền', 'donGia': 12600, 'thanhTien': 88200,
     'price_ref': 12600, 'loaiGia': 'KK trong nước', 'key_quality': 3},

    {'tenThuoc': 'Eulosan 50', 'soLuong': 13, 'donViTinh': 'Viên',
     'dangBaoChe': 'Viên nang cứng', 'donGia': 2100, 'thanhTien': 27300,
     'price_ref': 2100, 'loaiGia': 'KK trong nước', 'key_quality': 3},
]

print(f'  {"Tên thuốc":<25} {"SL":>4}  {"Đơn giá":>9}  {"price_ref":>9}  {"ML":>8}  {"Score":>6}')
print('  ' + '-'*75)
for item in test_items:
    result = predict_item(item, model, scaler, FEATURES)
    status = '🔴 ANOMALY' if result['is_anomaly_ml'] else '🟢 normal'
    print(f"  {item['tenThuoc']:<25} {item['soLuong']:>4}  "
          f"{item['donGia']:>9,}  {item['price_ref']:>9,}  "
          f"{status}  {result['anomaly_score']:>5.3f}")

## Bước tiếp theo

Sau khi chạy xong notebook này, bạn có **3 file** để copy vào project FastAPI:

```
pharma_model.zip
  ├── isolation_forest.pkl   → copy vào  model/
  ├── scaler.pkl             → copy vào  model/
  ├── features.pkl           → copy vào  model/
  └── metadata.json          → tham khảo
```

Trong `pipeline.py`, load model như sau:

```python
import pickle

with open('model/isolation_forest.pkl', 'rb') as f:
    model = pickle.load(f)
with open('model/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('model/features.pkl', 'rb') as f:
    FEATURES = pickle.load(f)
```

Rồi gọi `predict_item(item, model, scaler, FEATURES)` từ **Bước 8** cho mỗi dòng thuốc.